In [ ]:
# In[1]:

import pandas as pd

# Load CSVs (files are in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (per rule)
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_logs_df['ts'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True)

# Helper function to compute the required group stats
def group_stats(df, group_cols, value_col='value'):
    g = df.groupby(group_cols)[value_col]
    agg = g.agg(
        count='count',
        mean='mean',
        std='std',
        min='min',
        max='max',
        p50=lambda x: x.quantile(0.50),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99)
    ).reset_index()
    agg = agg.sort_values('count', ascending=False)
    return agg

# Metric summaries grouped by (cmdb_id, kpi_name)
metric_summary = group_stats(metric_df, ['cmdb_id', 'kpi_name'])
metric_summary_top20 = metric_summary.head(20)

# Trace summaries grouped by (cmdb_id, trace_name)
trace_summary = group_stats(trace_df, ['cmdb_id', 'trace_name'])
trace_summary_top20 = trace_summary.head(20)

# Log summaries grouped by (cmdb_id, log_name)
log_summary = group_stats(log_df, ['cmdb_id', 'log_name'])
log_summary_top20 = log_summary.head(20)

# Unique cmdb_id values across metric, trace, log
unique_cmdbs = sorted(
    set(metric_df['cmdb_id'].dropna().unique()).union(
        set(trace_df['cmdb_id'].dropna().unique()),
        set(log_df['cmdb_id'].dropna().unique())
    )
)
unique_cmdbs_df = pd.DataFrame({'cmdb_id': unique_cmdbs})

# Number of rows in error_logs.csv
error_logs_count = error_logs_df.shape[0]

# Incident window (UTC) and counts within window
start = pd.to_datetime("2024-01-23 20:35:49", utc=True)
end = pd.to_datetime("2024-01-23 21:05:49", utc=True)

metric_window_count = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] <= end)].shape[0]
trace_window_count = trace_df[(trace_df['ts'] >= start) & (trace_df['ts'] <= end)].shape[0]
log_window_count = log_df[(log_df['ts'] >= start) & (log_df['ts'] <= end)].shape[0]

# Return compact summaries (limited to 20 rows where applicable)
metric_summary_top20, trace_summary_top20, log_summary_top20, unique_cmdbs_df, error_logs_count, metric_window_count, trace_window_count, log_window_count

```
Out[1]:
```
```python
# Summarize results in plain English using existing variables from previous execution.
# This cell produces a concise textual summary string and displays it.

# Build top-5 summaries for metric/trace/log (use available top20 dataframes)
m_top = metric_summary_top20[['cmdb_id', 'kpi_name', 'count']].head(5)
t_top = trace_summary_top20[['cmdb_id', 'trace_name', 'count']].head(5)
l_top = log_summary_top20[['cmdb_id', 'log_name', 'count']].head(5)

lines = []
lines.append("Summary of telemetry analysis (plain English):")
lines.append("")
lines.append("Metric groups (top 5 by count):")
for r in m_top.itertuples(index=False):
    lines.append(f" - {r.cmdb_id} | {r.kpi_name} | count={int(r.count)}")

lines.append("")
lines.append("Trace groups (top 5 by count):")
for r in t_top.itertuples(index=False):
    lines.append(f" - {r.cmdb_id} | {r.trace_name} | count={int(r.count)}")

lines.append("")
lines.append("Log groups (top 5 by count):")
for r in l_top.itertuples(index=False):
    lines.append(f" - {r.cmdb_id} | {r.log_name} | count={int(r.count)}")

# Unique cmdb count and error logs count
unique_count = int(unique_cmdbs_df.shape[0])
lines.append("")
lines.append(f"Unique cmdb_id values across metric/trace/log: {unique_count}")

lines.append(f"Number of rows in error_logs.csv: {int(error_logs_count)}")

# Incident window counts
lines.append("")
lines.append("Rows within incident window 2024-01-23 20:35:49 to 2024-01-23 21:05:49 UTC:")
lines.append(f" - metric rows in window: {int(metric_window_count)}")
lines.append(f" - trace rows in window:  {int(trace_window_count)}")
lines.append(f" - log rows in window:    {int(log_window_count)}")

# Compact final note
lines.append("")
lines.append("Notes: top groups shown are the highest-count groups (counts shown). Most top groups have count=25 across the full series. error_logs.csv contains 0 rows.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count          mean           std           min           max           p50           p95           p99
357           ts-voucher-service      socket     25  4.208000e+00  8.255907e-01  4.000000e+00  8.000000e+00  4.000000e+00  4.960000e+00  7.328000e+00
0    ts-admin-basic-info-service         cpu     25  1.013133e+00  4.887291e-01  4.408533e-01  2.422903e+00  8.652821e-01  1.843079e+00  2.287632e+00
1    ts-admin-basic-info-service      diskio     25  5.177625e+04  4.437290e+03  4.050101e+04  6.196994e+04  5.158400e+04  5.770214e+04  6.098644e+04
2    ts-admin-basic-info-service       error     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3    ts-admin-basic-info-service  latency-50     25  1.782130e-02  4.476119e-04  1.750000e-02  1.912503e-02  1.767500e-02  1.874751e-02  1.905214e-02
4    ts-admin-basic-info-service  latency-90     25  2.633658e-02  6.524524e-03  2.350000e-02  5.186364e-02  2.382900e-02  3.793694e-02  4.871587e-02
318             ts-travel2-mongo         mem     25  7.087768e+07  5.353020e+05  7.018796e+07  7.164430e+07  7.060746e+07  7.163638e+07  7.164362e+07
319             ts-travel2-mongo      socket     25  7.000000e+00  0.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00
320           ts-travel2-service         cpu     25  6.838039e+00  2.913628e+00  4.104250e+00  1.466134e+01  5.873894e+00  1.262676e+01  1.424842e+01
321           ts-travel2-service      diskio     25  5.129823e+04  1.057582e+04  2.507183e+04  7.969210e+04  5.197728e+04  6.072119e+04  7.515146e+04
322           ts-travel2-service  latency-50     25  1.496813e-02  4.438402e-03  9.684236e-03  2.220292e-02  1.529711e-02  2.117275e-02  2.197026e-02
323           ts-travel2-service  latency-90     25  3.894027e-01  5.624178e-02  3.439978e-01  6.098285e-01  3.693158e-01  4.726212e-01  5.788215e-01
324           ts-travel2-service         mem     25  2.740962e+08  1.750810e+06  2.688454e+08  2.756615e+08  2.745758e+08  2.756365e+08  2.756602e+08
325           ts-travel2-service      socket     25  1.703553e+01  1.174761e+00  1.492000e+01  1.965000e+01  1.700000e+01  1.859000e+01  1.940200e+01
310            ts-travel-service      diskio     25  5.143191e+04  5.425964e+03  4.033101e+04  5.933099e+04  5.078719e+04  5.895876e+04  5.929873e+04
311            ts-travel-service  latency-50     25  1.852855e-02  1.726630e-03  1.588128e-02  2.206571e-02  1.837080e-02  2.180304e-02  2.203980e-02
312            ts-travel-service  latency-90     25  1.554177e-01  6.993354e-02  8.946603e-02  3.427165e-01  1.414423e-01  3.065929e-01  3.368980e-01
313            ts-travel-service         mem     25  2.615362e+08  2.554669e+06  2.527560e+08  2.647005e+08  2.624178e+08  2.631549e+08  2.643300e+08
314            ts-travel-service      socket     25  1.634137e+01  1.294384e+00  1.420000e+01  1.936667e+01  1.596667e+01  1.880667e+01  1.924667e+01
315            ts-travel-service    workload     25  9.218559e+00  1.734046e+00  4.204417e+00  1.164125e+01  9.312283e+00  1.153093e+01  1.162203e+01,                 cmdb_id                                        trace_name  count         mean         std         min          max          p50          p95          p99
728  ts-travel2-service          trace.from_ts-seat-service.duration_mean     25     0.012042    0.003945    0.008668     0.022044     0.010353     0.021083     0.021942
729  ts-travel2-service           trace.from_ts-seat-service.duration_p95     25     0.027024    0.024273    0.010652     0.079987     0.015681     0.079788     0.079981
730  ts-travel2-service             trace.from_ts-seat-service.error_rate     25     0.000000    0.000000    0.000000     0.000000     0.000000     0.000000     0.000000
731  ts-travel2-service              trace.from_ts-seat-service.row_count     25   297.280000   53.708721   56.000000   342.000000   304.000000   336.800000   340.800000
732  ts-travel2-service       trace.from_ts-travel2-service.duration_mean     25     0.010945    0.002423    0.008869     0.016704     0.009769     0.015766     0.016486
733  ts-travel2-service        trace.from_ts-travel2-service.duration_p95     25     0.049678    0.009868    0.042281     0.074617     0.045519     0.071122     0.073815
734  ts-travel2-service          trace.from_ts-travel2-service.error_rate     25     0.000000    0.000000    0.000000     0.000000     0.000000     0.000000     0.000000
735  ts-travel2-service           trace.from_ts-travel2-service.row_count     25  2420.920000  427.139021  484.000000  2735.000000  2485.000000  2722.600000  2732.600000
720  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean     25     0.073729    0.026617    0.054039     0.151558     0.064721     0.138890     0.150824
721  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95     25     0.114632    0.066175    0.059696     0.277982     0.093186     0.249623     0.272202
722  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate     25     0.000000    0.000000    0.000000     0.000000     0.000000     0.000000     0.000000
723  ts-travel2-service      trace.from_ts-admin-travel-service.row_count     25     8.800000    1.707825    2.000000    11.000000     9.000000    10.800000    11.000000
712   ts-travel-service          trace.to_ts-travel-service.duration_mean     25     0.012154    0.003327    0.008941     0.021347     0.010550     0.018953     0.020929
713   ts-travel-service           trace.to_ts-travel-service.duration_p95     25     0.055799    0.014680    0.044605     0.098644     0.048220     0.086614     0.096459
714   ts-travel-service             trace.to_ts-travel-service.error_rate     25     0.000000    0.000000    0.000000     0.000000     0.000000     0.000000     0.000000
715   ts-travel-service              trace.to_ts-travel-service.row_count     25  1485.240000  383.098364  270.000000  1975.000000  1566.000000  1927.400000  1967.800000
716  ts-travel2-service                     trace.from_root.duration_mean     25     0.330156    0.069915    0.267025     0.506593     0.301186     0.465702     0.497884
717  ts-travel2-service                      trace.from_root.duration_p95     25     0.504995    0.213198    0.292470     0.979138     0.445159     0.925380     0.967981
718  ts-travel2-service                        trace.from_root.error_rate     25     0.000000    0.000000    0.000000     0.000000     0.000000     0.000000     0.000000
719  ts-travel2-service                         trace.from_root.row_count     25    70.880000   11.211304   20.000000    79.000000    73.000000    78.800000    79.000000,                         cmdb_id         log_name  count     mean         std  min   max     p50     p95      p99
0   ts-admin-basic-info-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
1   ts-admin-basic-info-service    log.row_count     25    26.44    4.882622    7    32    28.0    30.8    31.76
2       ts-admin-travel-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
3       ts-admin-travel-service    log.row_count     25    26.40    5.082650    6    33    27.0    31.6    32.76
21           ts-consign-service    log.row_count     25    42.28   16.722041    7    77    42.0    63.0    73.64
20           ts-consign-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
13             ts-basic-service    log.row_count     25  2366.12  463.641952  460  2807  2468.0  2749.0  2794.52
12             ts-basic-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
11              ts-auth-service    log.row_count     25   308.32   52.407633   70   350   320.0   338.8   347.60
10              ts-auth-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
61             ts-price-service    log.row_count     25   260.00   52.003205   50   310   270.0   305.2   309.04
60             ts-price-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
64             ts-route-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
65             ts-route-service    log.row_count     25   963.88  177.180774  196  1112  1004.0  1093.2  1107.68
67              ts-seat-service    log.row_count     25  1772.96  356.681870  322  2114  1843.0  2076.8  2105.12
66              ts-seat-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
47             ts-order-service    log.row_count     25   371.48  100.160338   58   504   392.0   494.0   502.32
46             ts-order-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00
45       ts-order-other-service    log.row_count     25   415.60   77.521502   77   500   427.0   488.0   497.84
44       ts-order-other-service  log.error_count     25     0.00    0.000000    0     0     0.0     0.0     0.00,                          cmdb_id
0                           root
1    ts-admin-basic-info-service
2         ts-admin-order-service
3         ts-admin-route-service
4        ts-admin-travel-service
..                           ...
64                 ts-user-mongo
65               ts-user-service
66  ts-verification-code-service
67              ts-voucher-mysql
68            ts-voucher-service

[69 rows x 1 columns], 0, 8948, 18760, 2192)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing loaded DataFrames if present; otherwise load from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    trace_df = pd.read_csv("trace.csv")
    log_df = pd.read_csv("log.csv")
else:
    # ensure trace_df and log_df exist in workspace
    try:
        trace_df
    except NameError:
        trace_df = pd.read_csv("trace.csv")
    try:
        log_df
    except NameError:
        log_df = pd.read_csv("log.csv")

# Parse timestamps to UTC datetimes as required
for df in (metric_df, trace_df, log_df):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-23 20:35:49+00:00", utc=True)
end = pd.to_datetime("2024-01-23 21:05:49+00:00", utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

def compute_thresholds_and_window(df, cmdb_col, name_col, source_label):
    # Filter to candidate services (we only analyze these services)
    df_cand = df[df[cmdb_col].isin(candidates)].copy()
    if df_cand.empty:
        return pd.DataFrame(columns=[
            'source','cmdb_id','name','full_count','p95','window_count','anomaly_count','earliest_anomaly_timestamp'
        ])
    # Global thresholds (full series for these groups)
    grouped = df_cand.groupby([cmdb_col, name_col])['value']
    thresholds = grouped.agg(
        full_count='count',
        p50=lambda x: x.quantile(0.50),
        p95=lambda x: x.quantile(0.95)
    ).reset_index().rename(columns={cmdb_col: 'cmdb_id', name_col: 'name'})
    # Compute window statistics
    window_df = df_cand[(df_cand['ts'] >= start) & (df_cand['ts'] <= end)].copy()
    if window_df.empty:
        # No window rows for any group: window_count=0, anomaly_count=0, earliest_anomaly_timestamp=null
        thresholds['window_count'] = 0
        thresholds['anomaly_count'] = 0
        thresholds['earliest_anomaly_timestamp'] = None
    else:
        # compute window counts per group
        win_grp = window_df.groupby(['cmdb_id', 'name'] if 'name' in window_df.columns else [cmdb_col, name_col])
        # but window_df currently has original name_col, so ensure consistent columns
        window_df = window_df.rename(columns={cmdb_col: 'cmdb_id', name_col: 'name'})
        win_grp = window_df.groupby(['cmdb_id', 'name'])
        win_stats = win_grp.agg(window_count=('value', 'count')).reset_index()
        # For anomaly_count and earliest timestamp we need to compare value >= p95 (global)
        # Merge thresholds to window rows to evaluate anomaly
        merged_window = window_df.merge(thresholds[['cmdb_id','name','p95']], on=['cmdb_id','name'], how='left')
        # anomaly rows
        merged_window['is_anomaly'] = merged_window['value'] >= merged_window['p95']
        # compute anomaly counts and earliest anomaly timestamp
        anomaly_agg = merged_window[merged_window['is_anomaly']].groupby(['cmdb_id','name']).agg(
            anomaly_count=('is_anomaly','sum'),
            earliest_anom_ts=('ts','min')
        ).reset_index()
        # Merge window_count and anomaly aggregates into thresholds
        merged = thresholds.merge(win_stats, on=['cmdb_id','name'], how='left')
        merged = merged.merge(anomaly_agg, on=['cmdb_id','name'], how='left')
        merged['window_count'] = merged['window_count'].fillna(0).astype(int)
        merged['anomaly_count'] = merged['anomaly_count'].fillna(0).astype(int)
        # earliest_anom_ts to ISO string or None
        merged['earliest_anomaly_timestamp'] = merged['earliest_anom_ts'].apply(
            lambda x: x.isoformat() if pd.notna(x) else None
        )
        merged = merged.drop(columns=['earliest_anom_ts'])
        thresholds = merged

    # Build final DataFrame with required columns
    out = thresholds[['cmdb_id','name','full_count','p95','window_count','anomaly_count','earliest_anomaly_timestamp']].copy()
    out.insert(0, 'source', source_label)
    # Ensure types
    out['p95'] = pd.to_numeric(out['p95'])
    out['full_count'] = out['full_count'].astype(int)
    out['window_count'] = out['window_count'].astype(int)
    out['anomaly_count'] = out['anomaly_count'].astype(int)
    return out

# Compute for each source
metric_res = compute_thresholds_and_window(metric_df, 'cmdb_id', 'kpi_name', 'metric')
trace_res  = compute_thresholds_and_window(trace_df,  'cmdb_id', 'trace_name', 'trace')
log_res    = compute_thresholds_and_window(log_df,    'cmdb_id', 'log_name',   'log')

# Combine results
combined = pd.concat([metric_res, trace_res, log_res], ignore_index=True)

# Sort by anomaly_count desc then window_count desc, limit to top 50
combined_sorted = combined.sort_values(['anomaly_count','window_count'], ascending=[False, False]).head(50)

# Rows where anomaly_count > 0 (limit 50)
anomalies_only = combined_sorted[combined_sorted['anomaly_count'] > 0].head(50)

# Display compact tables (selected columns). According to rules, return variables.
combined_sorted[['source','cmdb_id','name','full_count','p95','window_count','anomaly_count','earliest_anomaly_timestamp']], anomalies_only[['source','cmdb_id','name','full_count','p95','window_count','anomaly_count','earliest_anomaly_timestamp']]

```
Out[2]:
```
```python
# Summarize anomaly findings for the five candidate services using variables from prior execution.
# This cell builds a concise plain-English summary string and displays it.

# Use existing dataframes: combined (all groups), combined_sorted (top50), anomalies_only (anomaly_count>0 subset)
try:
    total_groups = int(combined.shape[0])
    total_reported = int(combined_sorted.shape[0])
    anomaly_groups = int(anomalies_only.shape[0])
except NameError:
    total_groups = 0
    total_reported = 0
    anomaly_groups = 0

# Count how many of the anomaly groups come from trace error_rate KPIs
trace_error_rate_mask = (anomalies_only['source'] == 'trace') & (anomalies_only['name'].str.contains('error_rate', na=False))
trace_error_rate_count = int(anomalies_only[trace_error_rate_mask].shape[0])

# Prepare example rows (up to 10)
examples = []
for r in anomalies_only.head(10).itertuples(index=False):
    examples.append(f"- {r.source} | {r.cmdb_id} | {r.name} | p95={r.p95:.6g} | window_count={r.window_count} | anomaly_count={r.anomaly_count} | first={r.earliest_anomaly_timestamp}")

lines = []
lines.append("Summary of anomaly analysis for candidate services (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service):")
lines.append("")
lines.append(f"Total groups analyzed across metric/trace/log: {total_groups} (showing top {total_reported} sorted by anomaly_count/window_count).")
lines.append(f"Groups with anomaly_count > 0: {anomaly_groups}.")
lines.append("")
lines.append("Key observation:")
lines.append(f" - A large portion of anomaly groups are trace 'error_rate' metrics: {trace_error_rate_count} groups. Many of these have p95 = 0.0, which causes all non-negative values in the incident window to be flagged as anomalies (anomaly_count equals the full window_count, e.g., 25).")
lines.append("")
lines.append("Metric-level anomalies (examples):")
# show metric examples specifically if present
metric_examples = anomalies_only[anomalies_only['source']=='metric'].head(5)
if not metric_examples.empty:
    for r in metric_examples.itertuples(index=False):
        lines.append(f" - metric | {r.cmdb_id} | {r.name} | p95={r.p95:.6g} | window_count={r.window_count} | anomaly_count={r.anomaly_count} | first={r.earliest_anomaly_timestamp}")
else:
    lines.append(" - (no metric anomalies in top results)")

lines.append("")
lines.append("Trace/log-level anomaly examples (up to 10):")
if examples:
    lines.extend(examples)
else:
    lines.append(" - (none)")

lines.append("")
lines.append("Conclusion and recommendation:")
lines.append(" - Trace 'error_rate' groups with p95 == 0 are producing many trivial anomalies; verify whether p95=0 is meaningful for these KPIs or adjust anomaly logic (e.g., require > p95 and/or a minimum absolute threshold).")
lines.append(" - Metric anomalies are fewer and localized (e.g., ts-auth-service shows several metric KPIs with small anomaly counts). Inspect the listed metric KPIs and the raw values/timestamps for root cause investigation.")
lines.append("")
lines.append("If you want, I can: return the full anomaly rows for inspection, show raw values for a specific group, or recompute anomalies using a stricter rule (e.g., value > p95 and value > epsilon).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source           cmdb_id                                    name  full_count           p95  window_count  anomaly_count earliest_anomaly_timestamp
37   trace   ts-auth-service              trace.from_root.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
41   trace   ts-auth-service   trace.from_ts-auth-service.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
45   trace   ts-auth-service     trace.to_ts-auth-service.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
49   trace  ts-order-service              trace.from_root.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
57   trace  ts-order-service  trace.from_ts-order-service.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
..     ...               ...                                     ...         ...           ...           ...            ...                        ...
0   metric   ts-auth-service                                     cpu          25  1.735118e+01            25              2  2024-01-23T20:38:00+00:00
1   metric   ts-auth-service                                  diskio          25  6.063410e+04            25              2  2024-01-23T20:39:00+00:00
2   metric   ts-auth-service                              latency-50          25  8.382439e-01            25              2  2024-01-23T20:52:00+00:00
3   metric   ts-auth-service                              latency-90          25  1.908444e+00            25              2  2024-01-23T20:52:00+00:00
4   metric   ts-auth-service                                     mem          25  2.478206e+08            25              2  2024-01-23T20:58:00+00:00

[50 rows x 8 columns],     source           cmdb_id                                    name  full_count           p95  window_count  anomaly_count earliest_anomaly_timestamp
37   trace   ts-auth-service              trace.from_root.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
41   trace   ts-auth-service   trace.from_ts-auth-service.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
45   trace   ts-auth-service     trace.to_ts-auth-service.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
49   trace  ts-order-service              trace.from_root.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
57   trace  ts-order-service  trace.from_ts-order-service.error_rate          25  0.000000e+00            25             25  2024-01-23T20:38:00+00:00
..     ...               ...                                     ...         ...           ...           ...            ...                        ...
0   metric   ts-auth-service                                     cpu          25  1.735118e+01            25              2  2024-01-23T20:38:00+00:00
1   metric   ts-auth-service                                  diskio          25  6.063410e+04            25              2  2024-01-23T20:39:00+00:00
2   metric   ts-auth-service                              latency-50          25  8.382439e-01            25              2  2024-01-23T20:52:00+00:00
3   metric   ts-auth-service                              latency-90          25  1.908444e+00            25              2  2024-01-23T20:52:00+00:00
4   metric   ts-auth-service                                     mem          25  2.478206e+08            25              2  2024-01-23T20:58:00+00:00

[50 rows x 8 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp parsed to UTC datetimes per rules
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate services and incident window
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
start = pd.to_datetime("2024-01-23 20:35:49+00:00", utc=True)
end   = pd.to_datetime("2024-01-23 21:05:49+00:00", utc=True)

# 1) Compute global p95 per (cmdb_id, kpi_name) using full series (no window filtering)
metric_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()
thresholds = metric_cand.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# 2) Filter metric.csv to the incident window
window_df = metric_cand[(metric_cand['ts'] >= start) & (metric_cand['ts'] <= end)].copy()
window_df = window_df.sort_values(['cmdb_id','kpi_name','ts'])

# Merge p95 into window rows (ensures thresholds computed on full series are used)
window_df = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# 3) For each (cmdb_id, kpi_name), find consecutive runs where value >= p95
window_df['cond'] = window_df['value'] >= window_df['p95']

def extract_runs(g):
    g = g.sort_values('ts').reset_index(drop=True)
    if g.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','p95','fault_start_iso','fault_end_iso','length_in_minutes','max_value_in_run'
        ])
    diff_secs = g['ts'].diff().dt.total_seconds().fillna(60.0)
    prev_cond = g['cond'].shift(fill_value=False)
    start_of_run = g['cond'] & ((diff_secs != 60.0) | (~prev_cond))
    run_id = start_of_run.cumsum()
    run_id = run_id.where(g['cond'], 0)
    g['run_id'] = run_id
    runs = g[g['run_id'] > 0].groupby('run_id').agg(
        cmdb_id=('cmdb_id','first'),
        kpi_name=('kpi_name','first'),
        p95=('p95','first'),
        fault_start_ts=('ts','min'),
        fault_end_ts=('ts','max'),
        length_in_minutes=('ts','count'),
        max_value_in_run=('value','max')
    ).reset_index(drop=True)
    if not runs.empty:
        runs['fault_start_iso'] = runs['fault_start_ts'].apply(lambda x: x.isoformat())
        runs['fault_end_iso'] = runs['fault_end_ts'].apply(lambda x: x.isoformat())
        runs = runs[['cmdb_id','kpi_name','p95','fault_start_iso','fault_end_iso','length_in_minutes','max_value_in_run']]
    return runs

grouped = window_df.groupby(['cmdb_id','kpi_name'], sort=False)
runs_list = [extract_runs(grp) for _, grp in grouped]
faults = pd.concat(runs_list, ignore_index=True) if runs_list else pd.DataFrame(columns=[
    'cmdb_id','kpi_name','p95','fault_start_iso','fault_end_iso','length_in_minutes','max_value_in_run'
])

# 4) Keep only runs with length_in_minutes >= 2, sort, limit top 20
faults_filtered = faults[faults['length_in_minutes'] >= 2].copy()
faults_top20 = faults_filtered.sort_values(['length_in_minutes','max_value_in_run'], ascending=[False, False]).head(20)

# Present compact table only
faults_top20[['cmdb_id','kpi_name','p95','fault_start_iso','fault_end_iso','length_in_minutes','max_value_in_run']]

```
Out[3]:
```
```python
# Build a plain-English summary string from the previously computed faults_top20 DataFrame
try:
    df = faults_top20[['cmdb_id','kpi_name','p95','fault_start_iso','fault_end_iso','length_in_minutes','max_value_in_run']].reset_index(drop=True)
except NameError:
    summary = "No fault runs found (faults_top20 not available)."
else:
    lines = []
    lines.append("Detected fault runs (metric p95-based) within incident window for candidate services:")
    lines.append("")
    if df.empty:
        lines.append(" - No runs of length >= 2 minutes were found.")
    else:
        for r in df.itertuples(index=False):
            lines.append(f" - Service: {r.cmdb_id} | KPI: {r.kpi_name} | p95={r.p95:.6g} | start={r.fault_start_iso} | end={r.fault_end_iso} | length_min={int(r.length_in_minutes)} | max_value={r.max_value_in_run:.6g}")
        lines.append("")
        lines.append(f"Total runs returned: {len(df)} (filtered to length_in_minutes >= 2).")
        lines.append("Sorted by length desc then max_value desc; up to 20 rows returned.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name        p95            fault_start_iso              fault_end_iso  length_in_minutes  max_value_in_run
37   ts-route-service    workload  36.096813  2024-01-23T20:48:00+00:00  2024-01-23T20:49:00+00:00                  2         37.154833
0     ts-auth-service         cpu  17.351180  2024-01-23T20:38:00+00:00  2024-01-23T20:39:00+00:00                  2         18.985080
50  ts-travel-service         cpu  11.058206  2024-01-23T20:39:00+00:00  2024-01-23T20:40:00+00:00                  2         14.519398
43   ts-train-service  latency-90   0.016126  2024-01-23T20:40:00+00:00  2024-01-23T20:41:00+00:00                  2          0.034529
17   ts-order-service  latency-50   0.008516  2024-01-23T20:38:00+00:00  2024-01-23T20:39:00+00:00                  2          0.009853
42   ts-train-service  latency-50   0.007679  2024-01-23T20:38:00+00:00  2024-01-23T20:39:00+00:00                  2          0.007775
30   ts-route-service  latency-50   0.004299  2024-01-23T20:38:00+00:00  2024-01-23T20:39:00+00:00                  2          0.005013```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Parse timestamps to UTC datetimes (rule)
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services and incident window
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
start = pd.to_datetime("2024-01-23 20:35:49+00:00", utc=True)
end   = pd.to_datetime("2024-01-23 21:05:49+00:00", utc=True)

# Restrict to candidate services for threshold calculations
trace_cand = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

# 1) Compute global p95 per (cmdb_id, trace_name) using full series (do NOT filter by window)
p95_df = trace_cand.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# 2) Filter trace rows to the incident window
window_df = trace_cand[(trace_cand['ts'] >= start) & (trace_cand['ts'] <= end)].copy()
# Ensure consistent sorting
window_df = window_df.sort_values(['cmdb_id','trace_name','ts'])

# 3) Compute required aggregates within the window
# window_count per group
win_cnt = window_df.groupby(['cmdb_id','trace_name']).size().reset_index(name='window_count')

# nonzero_error_count and earliest_nonzero_error_ts (trace_name contains 'error_rate' and value > 0)
err_mask = window_df['trace_name'].str.contains('error_rate', na=False) & (window_df['value'] > 0)
err_df = window_df[err_mask].copy()
if not err_df.empty:
    err_agg = err_df.groupby(['cmdb_id','trace_name']).agg(
        nonzero_error_count=('value','count'),
        earliest_nonzero_error_ts=('ts','min')
    ).reset_index()
    err_agg['earliest_nonzero_error_ts'] = err_agg['earliest_nonzero_error_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())
else:
    err_agg = pd.DataFrame(columns=['cmdb_id','trace_name','nonzero_error_count','earliest_nonzero_error_ts'])

# duration_anomaly_count and earliest_duration_anomaly_ts:
# Need to compare to global p95; merge p95 into window rows first
window_with_p95 = window_df.merge(p95_df, on=['cmdb_id','trace_name'], how='left')

dur_mask = window_with_p95['trace_name'].str.contains('duration_mean', na=False) & (window_with_p95['value'] >= window_with_p95['p95'])
dur_df = window_with_p95[dur_mask].copy()
if not dur_df.empty:
    dur_agg = dur_df.groupby(['cmdb_id','trace_name']).agg(
        duration_anomaly_count=('value','count'),
        earliest_duration_anomaly_ts=('ts','min')
    ).reset_index()
    dur_agg['earliest_duration_anomaly_ts'] = dur_agg['earliest_duration_anomaly_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())
else:
    dur_agg = pd.DataFrame(columns=['cmdb_id','trace_name','duration_anomaly_count','earliest_duration_anomaly_ts'])

# 4) Combine results into compact table
result = p95_df.merge(win_cnt, on=['cmdb_id','trace_name'], how='left')
result = result.merge(err_agg, on=['cmdb_id','trace_name'], how='left')
result = result.merge(dur_agg, on=['cmdb_id','trace_name'], how='left')

# Fill missing numeric counts with 0 and timestamps with None
result['window_count'] = result['window_count'].fillna(0).astype(int)
result['nonzero_error_count'] = result['nonzero_error_count'].fillna(0).astype(int)
result['duration_anomaly_count'] = result['duration_anomaly_count'].fillna(0).astype(int)
result['earliest_nonzero_error_ts'] = result['earliest_nonzero_error_ts'].where(result['earliest_nonzero_error_ts'].notna(), None)
result['earliest_duration_anomaly_ts'] = result['earliest_duration_anomaly_ts'].where(result['earliest_duration_anomaly_ts'].notna(), None)

# Select and order columns as requested
result_all = result[['cmdb_id','trace_name','p95','window_count',
                     'nonzero_error_count','earliest_nonzero_error_ts',
                     'duration_anomaly_count','earliest_duration_anomaly_ts']].copy()

# Sort by nonzero_error_count desc then duration_anomaly_count desc, limit to top 50
result_all = result_all.sort_values(['nonzero_error_count','duration_anomaly_count'], ascending=[False, False]).head(50)

# Subset where nonzero_error_count > 0 (limit 50)
result_nonzero = result_all[result_all['nonzero_error_count'] > 0].head(50)

# Return the compact tables (DataFrames)
result_all, result_nonzero

```
Out[4]:
```
```python
# Summarize trace analysis results for the five candidate services using existing DataFrames result_all and result_nonzero.
try:
    df = result_all
    df_nonzero = result_nonzero
except NameError:
    summary = "Trace analysis results not available in the current session."
else:
    total_groups = int(df.shape[0])
    total_nonzero_error_groups = int(df_nonzero.shape[0])
    # groups with duration anomalies
    dur_df = df[df['duration_anomaly_count'] > 0].copy()
    dur_count = int(dur_df.shape[0])
    # examples (up to 6)
    examples = []
    for r in dur_df.head(6).itertuples(index=False):
        examples.append(f"{r.cmdb_id} | {r.trace_name} | duration_anomaly_count={r.duration_anomaly_count} | first={r.earliest_duration_anomaly_ts}")
    # Build summary
    lines = [
        "Summary of trace.csv analysis for candidate services (window 2024-01-23 20:35:49 to 21:05:49 UTC):",
        "",
        f"- Total (cmdb_id, trace_name) groups evaluated: {total_groups}.",
        f"- Groups with nonzero trace 'error_rate' within the window: {total_nonzero_error_groups} (no such groups returned).",
        "",
        f"- Groups with duration_mean >= global p95 inside the window: {dur_count}.",
        "  Examples of duration anomalies (service | trace_name | count | earliest_ts):"
    ]
    if examples:
        lines += ["   - " + e for e in examples]
    else:
        lines.append("   - (no duration anomalies found)")
    lines += [
        "",
        "Key conclusions:",
        " - No nonzero trace error_rate was observed for the candidate services in the incident window (nonzero_error_count=0 across returned rows).",
        " - Several trace duration_mean metrics exceeded their global p95 during the window (duration_anomaly_count > 0). These are the likely trace-level latency anomalies to inspect further (examples listed above).",
        "",
        "Recommendation: investigate the listed duration anomalies (start timestamps shown) for upstream/downstream latency and correlate with metric-level findings; error_rate traces did not show non-zero spikes."
    ]
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id                                         trace_name        p95  window_count  nonzero_error_count earliest_nonzero_error_ts  duration_anomaly_count earliest_duration_anomaly_ts
0    ts-auth-service                      trace.from_root.duration_mean   0.761869            25                    0                      None                       2    2024-01-23T20:55:00+00:00
4    ts-auth-service           trace.from_ts-auth-service.duration_mean   0.203146            25                    0                      None                       2    2024-01-23T20:55:00+00:00
8    ts-auth-service             trace.to_ts-auth-service.duration_mean   0.203146            25                    0                      None                       2    2024-01-23T20:55:00+00:00
12  ts-order-service                      trace.from_root.duration_mean   0.124281            25                    0                      None                       2    2024-01-23T21:00:00+00:00
16  ts-order-service  trace.from_ts-inside-payment-service.duration_...   0.018627            24                    0                      None                       2    2024-01-23T20:49:00+00:00
..               ...                                                ...        ...           ...                  ...                       ...                     ...                          ...
14  ts-order-service                         trace.from_root.error_rate   0.000000            25                    0                      None                       0                         None
15  ts-order-service                          trace.from_root.row_count  10.000000            25                    0                      None                       0                         None
17  ts-order-service  trace.from_ts-inside-payment-service.duration_p95   0.064513            24                    0                      None                       0                         None
18  ts-order-service    trace.from_ts-inside-payment-service.error_rate   0.000000            24                    0                      None                       0                         None
19  ts-order-service     trace.from_ts-inside-payment-service.row_count   8.850000            24                    0                      None                       0                         None

[50 rows x 8 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, p95, window_count, nonzero_error_count, earliest_nonzero_error_ts, duration_anomaly_count, earliest_duration_anomaly_ts]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd

# Use existing result_all from prior analysis; if not present, raise informative error.
try:
    result_all
except NameError:
    raise NameError("result_all DataFrame not found in session. Please run the prior trace analysis cell to produce result_all.")

# Filter rows with duration anomalies
filtered = result_all[result_all['duration_anomaly_count'] > 0].copy()

# Parse earliest_duration_anomaly_ts to datetime for accurate sorting (handle None safely)
filtered['earliest_dt'] = pd.to_datetime(filtered['earliest_duration_anomaly_ts'], utc=True)

# Compact table sorted by earliest_duration_anomaly_ts ascending, limit to top 20
filtered_table = filtered.sort_values('earliest_dt', ascending=True).head(20)[
    ['cmdb_id','trace_name','p95','window_count','duration_anomaly_count','earliest_duration_anomaly_ts']
].reset_index(drop=True)

# Per-service earliest duration anomaly timestamp (one row per cmdb_id)
per_service = (
    filtered.dropna(subset=['earliest_dt'])
    .groupby('cmdb_id', sort=True)['earliest_dt']
    .min()
    .reset_index()
)
per_service['earliest_duration_anomaly_ts'] = per_service['earliest_dt'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())
per_service = per_service[['cmdb_id','earliest_duration_anomaly_ts']].reset_index(drop=True)

# Return compact results (two DataFrames)
filtered_table, per_service

```
Out[5]:
```
```python
# Summarize the filtered trace duration anomalies in plain English using existing DataFrames.
try:
    filtered_table
    per_service
except NameError:
    summary = "Required results (filtered_table or per_service) are not available in the session."
else:
    lines = []
    lines.append("Trace duration anomaly summary (window 2024-01-23 20:35:49 to 21:05:49 UTC):")
    lines.append("")
    lines.append(f"- Total trace runs with duration_mean >= global p95 (length >= 1): {len(filtered_table)} (showing up to 20, sorted by earliest anomaly).")
    lines.append("")
    lines.append("Top detected trace duration anomalies (service | trace_name | p95 | window_count | duration_anomaly_count | earliest_ts):")
    for r in filtered_table.itertuples(index=False):
        lines.append(f" - {r.cmdb_id} | {r.trace_name} | p95={r.p95:.6g} | window_count={r.window_count} | duration_anomaly_count={r.duration_anomaly_count} | first={r.earliest_duration_anomaly_ts}")
    lines.append("")
    lines.append("Per-service earliest trace latency anomaly times (one row per service):")
    for r in per_service.itertuples(index=False):
        lines.append(f" - {r.cmdb_id} | earliest_duration_anomaly_ts={r.earliest_duration_anomaly_ts}")
    lines.append("")
    lines.append("Quick interpretation:")
    lines.append(" - ts-train-service and ts-travel-service show multiple trace.duration_mean anomalies starting at ~2024-01-23T20:38:00Z.")
    lines.append(" - ts-order-service anomaly earliest at ~2024-01-23T20:40:00Z.")
    lines.append(" - ts-route-service anomalies earliest at ~2024-01-23T20:41:00Z.")
    lines.append(" - ts-auth-service earliest duration anomaly occurs later at ~2024-01-23T20:55:00Z.")
    lines.append("")
    lines.append("Recommendation: investigate the listed trace spans and timestamps for ts-train-service and ts-travel-service first (they show the earliest and multiple duration anomalies), then correlate with metric-level anomalies around those times.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                        trace_name       p95  window_count  duration_anomaly_count earliest_duration_anomaly_ts
0    ts-train-service         trace.from_ts-basic-service.duration_mean  0.009336            25                       2    2024-01-23T20:38:00+00:00
1   ts-travel-service          trace.from_ts-food-service.duration_mean  0.021617            25                       2    2024-01-23T20:38:00+00:00
2   ts-travel-service  trace.from_ts-admin-travel-service.duration_mean  0.160456            25                       2    2024-01-23T20:38:00+00:00
3    ts-train-service       trace.from_ts-travel2-service.duration_mean  0.007732            25                       2    2024-01-23T20:38:00+00:00
4    ts-train-service                     trace.from_root.duration_mean  0.001344             1                       1    2024-01-23T20:38:00+00:00
5    ts-train-service        trace.from_ts-travel-service.duration_mean  0.007009            25                       2    2024-01-23T20:39:00+00:00
6   ts-travel-service      trace.from_ts-preserve-service.duration_mean  0.592395            24                       2    2024-01-23T20:39:00+00:00
7   ts-travel-service                     trace.from_root.duration_mean  1.079346            25                       2    2024-01-23T20:39:00+00:00
8    ts-train-service           trace.to_ts-train-service.duration_mean  0.001173            25                       2    2024-01-23T20:39:00+00:00
9   ts-travel-service            trace.to_ts-seat-service.duration_mean  0.090000            25                       2    2024-01-23T20:39:00+00:00
10   ts-train-service         trace.from_ts-train-service.duration_mean  0.001173            25                       2    2024-01-23T20:39:00+00:00
11  ts-travel-service          trace.to_ts-travel-service.duration_mean  0.018953            25                       2    2024-01-23T20:39:00+00:00
12  ts-travel-service          trace.from_ts-seat-service.duration_mean  0.027347            25                       2    2024-01-23T20:39:00+00:00
13  ts-travel-service        trace.from_ts-travel-service.duration_mean  0.018953            25                       2    2024-01-23T20:39:00+00:00
14  ts-travel-service      trace.to_ts-ticketinfo-service.duration_mean  0.032092            25                       2    2024-01-23T20:39:00+00:00
15  ts-travel-service           trace.to_ts-train-service.duration_mean  0.007009            25                       2    2024-01-23T20:39:00+00:00
16   ts-order-service        trace.from_ts-travel-service.duration_mean  0.008831            25                       2    2024-01-23T20:40:00+00:00
17  ts-travel-service           trace.to_ts-order-service.duration_mean  0.008831            25                       2    2024-01-23T20:40:00+00:00
18   ts-route-service        trace.from_ts-travel-service.duration_mean  0.004804            25                       2    2024-01-23T20:41:00+00:00
19   ts-route-service       trace.from_ts-travel2-service.duration_mean  0.004635            25                       2    2024-01-23T20:41:00+00:00,              cmdb_id earliest_duration_anomaly_ts
0    ts-auth-service    2024-01-23T20:55:00+00:00
1   ts-order-service    2024-01-23T20:40:00+00:00
2   ts-route-service    2024-01-23T20:41:00+00:00
3   ts-train-service    2024-01-23T20:38:00+00:00
4  ts-travel-service    2024-01-23T20:38:00+00:00)```
```

In [ ]:
# In[6]:

import pandas as pd
import re

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Parse timestamps to UTC datetimes
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Incident window
start = pd.to_datetime("2024-01-23 20:35:49+00:00", utc=True)
end   = pd.to_datetime("2024-01-23 21:05:49+00:00", utc=True)

# Substring filters (as provided)
substrs = [
    'to_ts-train-service', 'from_ts-train-service',
    'to_ts-travel-service', 'from_ts-travel-service',
    'to_ts-order-service', 'from_ts-travel-service',
    'to_ts-seat-service', 'to_ts-travel-service'
]
# build regex OR pattern (escape hyphens not needed)
pattern = "(" + "|".join(re.escape(s) for s in substrs) + ")"

# 1) Compute full-series p95 per (cmdb_id, trace_name) using entire trace_df
p95_df = trace_df.groupby(['cmdb_id','trace_name'], sort=False)['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# 2) Filter groups whose trace_name contains any of the substrings
p95_filtered = p95_df[p95_df['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# 3) Filter trace rows to incident window and to the filtered trace_name set
window_df = trace_df[(trace_df['ts'] >= start) & (trace_df['ts'] <= end)].copy()
# Keep only trace rows whose trace_name matches our substrings
window_df = window_df[window_df['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# Merge p95 into window rows (will align p95 computed on full series)
window_with_p95 = window_df.merge(p95_filtered[['cmdb_id','trace_name','p95']],
                                  on=['cmdb_id','trace_name'], how='right' if window_df.empty else 'left')

# Prepare a base result DataFrame: all groups in p95_filtered (even if no window rows)
result = p95_filtered.merge(
    # window counts per group (may be NaN)
    window_df.groupby(['cmdb_id','trace_name']).size().reset_index(name='window_count'),
    on=['cmdb_id','trace_name'],
    how='left'
)

result['window_count'] = result['window_count'].fillna(0).astype(int)

# For duration anomalies: consider only trace_name ending with 'duration_mean'
# We'll compute duration_anomaly_count and earliest_duration_anomaly_ts per group
# Merge window rows with p95 for comparison
if not window_df.empty:
    win_p95 = window_df.merge(p95_filtered[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='right' if window_df.empty else 'left')
else:
    win_p95 = pd.DataFrame(columns=list(window_df.columns) + ['p95'])

# Identify duration_mean rows where value >= p95
dur_mask = win_p95['trace_name'].str.endswith('duration_mean', na=False) & (win_p95['value'] >= win_p95['p95'])
dur_rows = win_p95[dur_mask].copy()

if not dur_rows.empty:
    dur_agg = dur_rows.groupby(['cmdb_id','trace_name']).agg(
        duration_anomaly_count=('value','count'),
        earliest_duration_anomaly_ts=('ts','min')
    ).reset_index()
    dur_agg['earliest_duration_anomaly_ts'] = dur_agg['earliest_duration_anomaly_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())
else:
    dur_agg = pd.DataFrame(columns=['cmdb_id','trace_name','duration_anomaly_count','earliest_duration_anomaly_ts'])

# Merge duration aggregates into result
result = result.merge(dur_agg, on=['cmdb_id','trace_name'], how='left')
result['duration_anomaly_count'] = result['duration_anomaly_count'].fillna(0).astype(int)
result['earliest_duration_anomaly_ts'] = result['earliest_duration_anomaly_ts'].where(result['earliest_duration_anomaly_ts'].notna(), None)

# Final select and sort by earliest_duration_anomaly_ts ascending (None/NaT go last)
result['earliest_dt'] = pd.to_datetime(result['earliest_duration_anomaly_ts'], utc=True)
result_sorted = result.sort_values(['earliest_dt'], ascending=[True]).head(50)

result_table = result_sorted[['cmdb_id','trace_name','p95','window_count','duration_anomaly_count','earliest_duration_anomaly_ts']].reset_index(drop=True)

# 4) Infer caller/callee pairs from trace_name strings in the filtered set
# Extract 'to_' or 'from_' token followed by service identifier (e.g., ts-travel-service)
def infer_pair(row):
    tn = row['trace_name']
    cmdb = row['cmdb_id']
    m = re.search(r'(?:to_|from_)(ts-[^\.]+)', tn)
    if not m:
        return None
    svc = m.group(1)
    if 'to_' in tn:
        caller = cmdb
        callee = svc
    elif 'from_' in tn:
        caller = svc
        callee = cmdb
    else:
        return None
    return (caller, callee)

pairs = result_table.apply(infer_pair, axis=1).dropna().unique()
# Build DataFrame of unique pairs
if len(pairs) > 0:
    relation_pairs = pd.DataFrame(pairs.tolist(), columns=['caller_service','callee_service'])
else:
    relation_pairs = pd.DataFrame(columns=['caller_service','callee_service'])

# Output compact tables
result_table, relation_pairs

```
Out[6]:
```
```python
# Build a plain-English summary from result_table and relation_pairs produced earlier.
try:
    rt = result_table
    rp = relation_pairs
except NameError:
    summary = "Result tables (result_table or relation_pairs) are not available in the session."
else:
    total_rows = int(rt.shape[0])
    earliest_ts = rt['earliest_duration_anomaly_ts'].dropna().min()
    # Determine top callee services implicated (most frequent callee in trace_name strings)
    # Extract callee tokens from trace_name (to_ts-<service> or from_ts-<service>)
    import re
    def extract_target(tn):
        m = re.search(r'(?:to_ts-|from_ts-)(ts-[^\.]+)', tn)
        return m.group(1) if m else None
    rt['callee'] = rt['trace_name'].apply(extract_target)
    callee_counts = rt['callee'].value_counts().head(10)
    # Sample inferred caller->callee pairs (up to 10)
    sample_pairs = rp.head(10).to_records(index=False).tolist()
    lines = []
    lines.append("Summary of downstream trace-duration anomalies (filtered traces targeting train/travel/order/seat services):")
    lines.append("")
    lines.append(f"- Number of (cmdb_id, trace_name) groups returned: {total_rows} (limited to top 50 sorted by earliest anomaly).")
    lines.append(f"- Earliest detected duration anomaly timestamp in this set: {earliest_ts}")
    lines.append("")
    lines.append("Key observations:")
    lines.append(" - Multiple trace.duration_mean anomalies appear very early in the window (around 2024-01-23T20:38:00Z).")
    lines.append(" - The anomalies predominantly involve calls targeting ts-train-service and ts-travel-service (many 'to_ts-train-service' and 'to_ts-travel-service' traces).")
    lines.append("")
    lines.append("Top implicated callees (examples, by frequency in the filtered set):")
    for svc, cnt in callee_counts.items():
        lines.append(f" - {svc}: {int(cnt)} trace groups")
    lines.append("")
    lines.append("Sample inferred caller -> callee relations (one row per inferred relation, up to 10):")
    for caller, callee in sample_pairs:
        lines.append(f" - {caller} -> {callee}")
    lines.append("")
    lines.append("Interpretation and recommendation:")
    lines.append(" - The earliest and repeated downstream latency anomalies point to ts-train-service and ts-travel-service as likely downstream hotspots starting ~20:38Z.")
    lines.append(" - Investigate these callee services' performance (their service stacks and resource metrics) and inspect the listed trace spans from the caller services for more context.")
    lines.append(" - Correlate these trace anomalies with metric-level spikes (CPU/memory/disk/latency) around the same timestamps to find root cause.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                 trace_name        p95  window_count  duration_anomaly_count earliest_duration_anomaly_ts
0                      root    trace.to_ts-train-service.duration_mean   0.001344             1                       1    2024-01-23T20:38:00+00:00
1        ts-travel2-service    trace.to_ts-train-service.duration_mean   0.007732            25                       2    2024-01-23T20:38:00+00:00
2   ts-admin-travel-service   trace.to_ts-travel-service.duration_mean   0.160456            25                       2    2024-01-23T20:38:00+00:00
3          ts-basic-service    trace.to_ts-train-service.duration_mean   0.009336            25                       2    2024-01-23T20:38:00+00:00
4           ts-food-service   trace.to_ts-travel-service.duration_mean   0.021617            25                       2    2024-01-23T20:38:00+00:00
..                      ...                                        ...        ...           ...                     ...                          ...
45          ts-food-service    trace.to_ts-travel-service.duration_p95   0.061051            25                       0                         None
46          ts-food-service      trace.to_ts-travel-service.error_rate   0.000000            25                       0                         None
47          ts-food-service       trace.to_ts-travel-service.row_count  27.000000            25                       0                         None
48         ts-order-service  trace.from_ts-travel-service.duration_p95   0.018886            25                       0                         None
49         ts-order-service    trace.from_ts-travel-service.error_rate   0.000000            25                       0                         None

[50 rows x 6 columns],                caller_service     callee_service
0                        root   ts-train-service
1          ts-travel2-service   ts-train-service
2     ts-admin-travel-service  ts-travel-service
3            ts-basic-service   ts-train-service
4             ts-food-service  ts-travel-service
..                        ...                ...
20        ts-preserve-service   ts-order-service
21        ts-security-service   ts-order-service
22  ts-inside-payment-service   ts-order-service
23           ts-order-service   ts-order-service
24                       root   ts-order-service

[25 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd

# Reuse existing DataFrames if present; otherwise load from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

# Ensure timestamps parsed to UTC datetimes per rule
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_logs_df['ts'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True)

# Focus services and incident window
services = ["ts-train-service", "ts-travel-service"]
start = pd.to_datetime("2024-01-23 20:35:49+00:00", utc=True)
end   = pd.to_datetime("2024-01-23 21:05:49+00:00", utc=True)

# 1) Compute global p95 per (cmdb_id, kpi_name) for the two services (full series)
metric_cand = metric_df[metric_df['cmdb_id'].isin(services)].copy()
p95_metric = metric_cand.groupby(['cmdb_id','kpi_name'], sort=False)['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# 2) Filter metric rows to incident window
metric_window = metric_cand[(metric_cand['ts'] >= start) & (metric_cand['ts'] <= end)].copy()
metric_window = metric_window.sort_values(['cmdb_id','kpi_name','ts'])

# Merge p95 into window rows
metric_window = metric_window.merge(p95_metric, on=['cmdb_id','kpi_name'], how='left')

# 3) For each group find consecutive runs where value >= p95; runs length >=2
metric_window['cond'] = metric_window['value'] >= metric_window['p95']

def extract_runs(g):
    g = g.sort_values('ts').reset_index(drop=True)
    if g.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','p95','fault_start_iso','fault_end_iso','length_in_minutes','max_value_in_run'
        ])
    # time difference in seconds between consecutive points
    diff_secs = g['ts'].diff().dt.total_seconds().fillna(60.0)
    prev_cond = g['cond'].shift(fill_value=False)
    # start a run when cond is True and either previous wasn't True or timestamps are not exactly consecutive (assume per-minute series -> 60s)
    start_of_run = g['cond'] & (~prev_cond | (diff_secs != 60.0))
    run_id = start_of_run.cumsum()
    # only keep run ids for rows where cond is True
    run_id = run_id.where(g['cond'], 0)
    g['run_id'] = run_id
    runs = g[g['run_id'] > 0].groupby('run_id', sort=False).agg(
        cmdb_id=('cmdb_id','first'),
        kpi_name=('kpi_name','first'),
        p95=('p95','first'),
        fault_start_ts=('ts','min'),
        fault_end_ts=('ts','max'),
        length_in_minutes=('ts','count'),
        max_value_in_run=('value','max')
    ).reset_index(drop=True)
    if not runs.empty:
        runs['fault_start_iso'] = runs['fault_start_ts'].apply(lambda x: x.isoformat())
        runs['fault_end_iso'] = runs['fault_end_ts'].apply(lambda x: x.isoformat())
        runs = runs[['cmdb_id','kpi_name','p95','fault_start_iso','fault_end_iso','length_in_minutes','max_value_in_run']]
    return runs

groups = metric_window.groupby(['cmdb_id','kpi_name'], sort=False)
runs_list = [extract_runs(grp) for _, grp in groups]
metric_faults = pd.concat(runs_list, ignore_index=True) if runs_list else pd.DataFrame(columns=[
    'cmdb_id','kpi_name','p95','fault_start_iso','fault_end_iso','length_in_minutes','max_value_in_run'
])

# Keep only runs with length_in_minutes >= 2
metric_faults_filtered = metric_faults[metric_faults['length_in_minutes'] >= 2].copy()

# Sort and limit top 20 rows
metric_faults_top20 = metric_faults_filtered.sort_values(['length_in_minutes','max_value_in_run'], ascending=[False, False]).head(20).reset_index(drop=True)

# --- Log analysis for the two services within the incident window ---
log_cand = log_df[log_df['cmdb_id'].isin(services)].copy()

# 1) full-series p95 per (cmdb_id, log_name)
p95_log = log_cand.groupby(['cmdb_id','log_name'], sort=False)['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# 2) window rows
log_window = log_cand[(log_cand['ts'] >= start) & (log_cand['ts'] <= end)].copy()

# window_count per (cmdb_id, log_name)
win_cnt_log = log_window.groupby(['cmdb_id','log_name']).size().reset_index(name='window_count')

# per-service sums/means for requested aggregates within window
# sum_of_log_error_count: sum of value where log_name == 'log.error_count' within window per service
err_sum = log_window[log_window['log_name']=='log.error_count'].groupby('cmdb_id')['value'].sum().reset_index().rename(columns={'value':'sum_of_log_error_count'})
# mean_log_row_count: mean of value where log_name == 'log.row_count' within window per service
row_mean = log_window[log_window['log_name']=='log.row_count'].groupby('cmdb_id')['value'].mean().reset_index().rename(columns={'value':'mean_log_row_count'})

# Merge p95 and window_count, and attach per-service aggregates (sum/mean) to each row
log_summary = p95_log.merge(win_cnt_log, on=['cmdb_id','log_name'], how='left')
log_summary['window_count'] = log_summary['window_count'].fillna(0).astype(int)
# attach per-service aggregates
log_summary = log_summary.merge(err_sum, on='cmdb_id', how='left')
log_summary = log_summary.merge(row_mean, on='cmdb_id', how='left')
log_summary['sum_of_log_error_count'] = log_summary['sum_of_log_error_count'].fillna(0)
log_summary['mean_log_row_count'] = log_summary['mean_log_row_count'].fillna(0)

# Keep compact columns and limit to top 20 rows (there should be few)
log_summary_table = log_summary[['cmdb_id','log_name','p95','window_count','sum_of_log_error_count','mean_log_row_count']].reset_index(drop=True).head(20)

# --- error_logs.csv rows for the two services within the incident window ---
errlogs_window = error_logs_df[ (error_logs_df['cmdb_id'].isin(services)) & (error_logs_df['ts'] >= start) & (error_logs_df['ts'] <= end) ]
error_logs_count_window = int(errlogs_window.shape[0])

# Return compact tables/values (variable names are the outputs)
metric_faults_top20, log_summary_table, error_logs_count_window

```
Out[7]:
```
```python
# Build a concise plain-English summary from the previously computed results.
try:
    metric_faults_top20
    log_summary_table
    error_logs_count_window
except NameError:
    summary = "Required results are not available in the session."
else:
    lines = []
    lines.append("Summary for ts-train-service and ts-travel-service during 2024-01-23 20:35:49–21:05:49 UTC:")
    lines.append("")
    lines.append("Metric-based fault runs (value >= global p95, runs >= 2 minutes):")
    if metric_faults_top20.empty:
        lines.append(" - No multi-minute metric fault runs detected.")
    else:
        for r in metric_faults_top20.itertuples(index=False):
            lines.append(f" - {r.cmdb_id} | KPI={r.kpi_name} | p95={r.p95:.6g} | start={r.fault_start_iso} | end={r.fault_end_iso} | length_min={int(r.length_in_minutes)} | max_value={r.max_value_in_run:.6g}")
    lines.append("")
    lines.append("Log summary for these services (within the same window):")
    if log_summary_table.empty:
        lines.append(" - No log groups found for the two services in the window.")
    else:
        for r in log_summary_table.itertuples(index=False):
            lines.append(f" - {r.cmdb_id} | {r.log_name} | p95={r.p95} | window_count={r.window_count} | sum_log.error_count={r.sum_of_log_error_count} | mean_log.row_count={r.mean_log_row_count:.2f}")
    lines.append("")
    lines.append(f"Number of rows in error_logs.csv for these two services within the window: {int(error_logs_count_window)}")
    lines.append("")
    lines.append("Quick interpretation and recommendation:")
    lines.append(" - Detected metric runs are short (2 minutes) for these services:")
    lines.append("     * ts-travel-service: CPU exceeded p95 at 2024-01-23T20:39:00Z–20:40:00Z (max ~14.52).")
    lines.append("     * ts-train-service: latency-90 and latency-50 exceeded p95 around 20:38–20:41Z (each 2-minute runs, small absolute latency values but above historical p95).")
    lines.append(" - Logs for ts-travel-service show no error_count in the window (sum = 0) and high log.row_count mean (~1670.96); ts-train-service had no log rows summarized here.")
    lines.append(" - No error_log entries were found for these services in the incident window.")
    lines.append("")
    lines.append("Next steps:")
    lines.append(" - Investigate ts-travel-service CPU spike at ~20:39Z and ts-train-service latency anomalies around ~20:38–20:41Z; correlate with upstream callers (trace anomalies) and any deployment or resource events.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id    kpi_name        p95            fault_start_iso              fault_end_iso  length_in_minutes  max_value_in_run
0  ts-travel-service         cpu  11.058206  2024-01-23T20:39:00+00:00  2024-01-23T20:40:00+00:00                  2         14.519398
1   ts-train-service  latency-90   0.016126  2024-01-23T20:40:00+00:00  2024-01-23T20:41:00+00:00                  2          0.034529
2   ts-train-service  latency-50   0.007679  2024-01-23T20:38:00+00:00  2024-01-23T20:39:00+00:00                  2          0.007775,              cmdb_id         log_name     p95  window_count  sum_of_log_error_count  mean_log_row_count
0  ts-travel-service  log.error_count     0.0            25                       0             1670.96
1  ts-travel-service    log.row_count  2159.0            25                       0             1670.96, 0)```
```

In [ ]:
# In[8]:

import pandas as pd

# Ensure metric_df is loaded in session
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Parse timestamps to UTC datetimes (idempotent)
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Parameters
service = 'ts-train-service'
start = pd.to_datetime("2024-01-23 20:35:49+00:00", utc=True)
end   = pd.to_datetime("2024-01-23 21:05:49+00:00", utc=True)

# 1) Compute global p95 per kpi_name for ts-train-service using full series (no window filtering)
p95_train = metric_df[metric_df['cmdb_id'] == service].groupby('kpi_name', sort=False)['value'].quantile(0.95).reset_index()
p95_train = p95_train.rename(columns={'value': 'p95'})

# 2) Filter metric rows to the incident window for the service
window_df = metric_df[(metric_df['cmdb_id'] == service) & (metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()
window_df = window_df.sort_values(['kpi_name','ts'])

# 3) For each kpi_name compute window_count, max_value_in_window, earliest timestamp where value >= p95, and count_ge_p95
# Merge p95 into window rows
window_with_p95 = window_df.merge(p95_train, on='kpi_name', how='left')

# Condition for >= p95
window_with_p95['ge_p95'] = window_with_p95['value'] >= window_with_p95['p95']

# Aggregations per kpi_name within the window
agg_window = window_with_p95.groupby('kpi_name', sort=False).agg(
    window_count=('value','count'),
    max_value_in_window=('value','max'),
    count_ge_p95=('ge_p95','sum'),
    earliest_ge_p95_ts=('ts', lambda x: x[window_with_p95.loc[x.index, 'ge_p95']].min() if any(window_with_p95.loc[x.index, 'ge_p95']) else pd.NaT)
).reset_index()

# Convert earliest timestamp to ISO string or None
agg_window['earliest_ge_p95_iso'] = agg_window['earliest_ge_p95_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else None)
agg_window = agg_window.drop(columns=['earliest_ge_p95_ts'])

# 4) Combine with full kpi list to include kpis that may have no window rows
result = p95_train.merge(agg_window, on='kpi_name', how='left')
result['window_count'] = result['window_count'].fillna(0).astype(int)
result['count_ge_p95'] = result['count_ge_p95'].fillna(0).astype(int)
result['max_value_in_window'] = result['max_value_in_window'].where(result['max_value_in_window'].notna(), None)

# Final compact table with requested columns
result_table = result[['kpi_name','p95','window_count','count_ge_p95','earliest_ge_p95_iso','max_value_in_window']].reset_index(drop=True)

# Limit is naturally <= 20 for this service; display the table variable
result_table

```
Out[8]:
```
```python
# Build a concise plain-English summary from the ts-train-service KPI table (result_table)
try:
    result_table
except NameError:
    summary = "Result table for ts-train-service not found in session."
else:
    lines = []
    lines.append("ts-train-service metric summary within 2024-01-23 20:35:49–21:05:49 UTC:")
    lines.append("")
    lines.append("For each KPI: global p95 (full series), number of points in the window, count >= p95 in window, earliest >=p95 timestamp, and max value observed in window.")
    lines.append("")
    for r in result_table.itertuples(index=False):
        lines.append(
            f" - {r.kpi_name}: p95={r.p95:.6g} | window_count={int(r.window_count)} | "
            f"count_ge_p95={int(r.count_ge_p95)} | earliest_ge_p95={r.earliest_ge_p95_iso} | max_in_window={r.max_value_in_window:.6g}"
        )
    lines.append("")
    lines.append("Key takeaways:")
    lines.append(" - All listed KPIs (cpu, diskio, latency-50, latency-90, mem, socket, workload) had 2 data points in the window >= their global p95.")
    lines.append(" - Earliest anomaly times by KPI (examples): latency-50 at 2024-01-23T20:38:00Z; cpu at 20:39:00Z; latency-90 at 20:40:00Z; workload at 20:43:00Z; socket at 20:49:00Z; mem at 20:55:00Z; diskio later at 20:59:00Z.")
    lines.append(" - These are short, repeated spikes (each KPI shows two points >= p95). The earliest metric anomalies align with trace-duration anomalies observed around ~20:38Z for ts-train-service.")
    lines.append("")
    lines.append("Recommendation:")
    lines.append(" - Investigate ts-train-service around 20:38–20:41Z first (latency and CPU spikes), correlate with trace spans and upstream callers; then check memory/disk activity later (~20:49–20:59Z).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name           p95  window_count  count_ge_p95        earliest_ge_p95_iso  max_value_in_window
0         cpu  1.060719e+01            25             2  2024-01-23T20:39:00+00:00         1.209963e+01
1      diskio  6.232204e+04            25             2  2024-01-23T20:59:00+00:00         6.793129e+04
2  latency-50  7.679148e-03            25             2  2024-01-23T20:38:00+00:00         7.774825e-03
3  latency-90  1.612648e-02            25             2  2024-01-23T20:40:00+00:00         3.452926e-02
4         mem  2.506241e+08            25             2  2024-01-23T20:55:00+00:00         2.512502e+08
5      socket  2.001333e+01            25             2  2024-01-23T20:49:00+00:00         2.032000e+01
6    workload  2.271748e+01            25             2  2024-01-23T20:43:00+00:00         2.345122e+01```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamps parsed to UTC datetimes (idempotent)
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Parameters
service = 'ts-train-service'
start = pd.to_datetime("2024-01-23 20:35:49+00:00", utc=True)
end   = pd.to_datetime("2024-01-23 21:05:49+00:00", utc=True)

# 1) Compute global p95 per kpi_name using full series (no window filtering)
p95_train = metric_df[metric_df['cmdb_id'] == service].groupby('kpi_name', sort=False)['value'].quantile(0.95).reset_index()
p95_train = p95_train.rename(columns={'value': 'p95'})

# 2) Filter metric rows to the incident window for the service
window_df = metric_df[(metric_df['cmdb_id'] == service) & (metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()
window_df = window_df.sort_values(['kpi_name','ts'])

# 3) Aggregations per kpi_name within the window
# Merge p95 into window rows to evaluate comparisons
window_with_p95 = window_df.merge(p95_train, on='kpi_name', how='left')

# Compute per-kpi aggregates in window
agg_win = window_with_p95.groupby('kpi_name', sort=False).agg(
    window_count=('value','count'),
    max_value_in_window=('value','max'),
    count_ge_p95=('value', lambda x: (x >= window_with_p95.loc[x.index, 'p95']).sum()),
    earliest_ge_p95_ts=('ts', lambda x: x[window_with_p95.loc[x.index, 'value'] >= window_with_p95.loc[x.index, 'p95']].min() if any(window_with_p95.loc[x.index, 'value'] >= window_with_p95.loc[x.index, 'p95']) else pd.NaT)
).reset_index()

# Convert earliest timestamp to ISO or None
agg_win['earliest_ge_p95_iso'] = agg_win['earliest_ge_p95_ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else None)
agg_win = agg_win.drop(columns=['earliest_ge_p95_ts'])

# 4) Combine with full kpi list to include kpis that may have no window rows
result = p95_train.merge(agg_win, on='kpi_name', how='left')

# Fill missing values for window aggregates
result['window_count'] = result['window_count'].fillna(0).astype(int)
result['count_ge_p95'] = result['count_ge_p95'].fillna(0).astype(int)
result['max_value_in_window'] = result['max_value_in_window'].where(result['max_value_in_window'].notna(), None)
result['earliest_ge_p95_iso'] = result['earliest_ge_p95_iso'].where(result['earliest_ge_p95_iso'].notna(), None)

# Compute percent_exceed = (max_value_in_window - p95) / p95, use NaN if p95 == 0 or max_value_in_window is None
def compute_percent(row):
    p95 = row['p95']
    mv = row['max_value_in_window']
    if mv is None or pd.isna(p95) or p95 == 0:
        return np.nan
    return (mv - p95) / p95

result['percent_exceed'] = result.apply(compute_percent, axis=1)

# Sort by percent_exceed desc (NaNs last), limit to <= 20 rows (all kpis for this service typically <=20)
result_table_sorted = result.sort_values('percent_exceed', ascending=False, na_position='last').reset_index(drop=True)

# Select and order requested columns
result_table_sorted = result_table_sorted[['kpi_name','p95','max_value_in_window','percent_exceed','earliest_ge_p95_iso','count_ge_p95']]

# Display compact table (variable output)
result_table_sorted

```
Out[9]:
```
```python
# Summarize ts-train-service KPI exceedance results in plain English using existing result_table_sorted.
try:
    result_table_sorted
except NameError:
    summary = "Result table for ts-train-service not found in session."
else:
    lines = []
    lines.append("ts-train-service — KPI exceedance summary (window 2024-01-23 20:35:49–21:05:49 UTC):")
    lines.append("")
    lines.append("For each KPI: global p95, max observed in window, percent_exceed = (max - p95)/p95, earliest time >= p95, count >= p95")
    lines.append("")
    for r in result_table_sorted.itertuples(index=False):
        pct = r.percent_exceed
        pct_str = f"{pct*100:.1f}%" if pd.notna(pct) else "NaN"
        lines.append(
            f" - {r.kpi_name}: p95={r.p95:.6g}, max={r.max_value_in_window:.6g}, percent_exceed={pct_str}, "
            f"earliest={r.earliest_ge_p95_iso}, count_ge_p95={int(r.count_ge_p95)}"
        )
    lines.append("")
    lines.append("Key takeaways:")
    lines.append(" - The largest relative spike is latency-90: +~114% over p95 (earliest 2024-01-23T20:40:00Z, 2 points >= p95).")
    lines.append(" - CPU shows a notable spike (+~14%, earliest 2024-01-23T20:39:00Z, 2 points).")
    lines.append(" - Disk I/O shows a later spike (+~9%, earliest 2024-01-23T20:59:00Z).")
    lines.append(" - Other KPIs (workload, socket, latency-50, mem) show small relative increases (1–3%) with two points each >= p95.")
    lines.append("")
    lines.append("Interpretation and recommended focus:")
    lines.append(" - Investigate ts-train-service around ~20:38–20:41Z for latency (latency-50/90) and CPU anomalies first.")
    lines.append(" - Then check later resource activity (memory/disk I/O) around ~20:49–20:59Z for possible secondary effects.")
    lines.append("")
    lines.append("All KPIs listed had 2 samples in the window >= their historical p95 (count_ge_p95 = 2).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name           p95  max_value_in_window  percent_exceed        earliest_ge_p95_iso  count_ge_p95
0  latency-90  1.612648e-02         3.452926e-02        1.141153  2024-01-23T20:40:00+00:00             2
1         cpu  1.060719e+01         1.209963e+01        0.140701  2024-01-23T20:39:00+00:00             2
2      diskio  6.232204e+04         6.793129e+04        0.090004  2024-01-23T20:59:00+00:00             2
3    workload  2.271748e+01         2.345122e+01        0.032298  2024-01-23T20:43:00+00:00             2
4      socket  2.001333e+01         2.032000e+01        0.015323  2024-01-23T20:49:00+00:00             2
5  latency-50  7.679148e-03         7.774825e-03        0.012459  2024-01-23T20:38:00+00:00             2
6         mem  2.506241e+08         2.512502e+08        0.002498  2024-01-23T20:55:00+00:00             2```
```